# Student Success and Placement Analytics

This notebook consolidates the complete student analytics workflow: data loading, cleaning, exploratory analysis, business KPI calculation, visualization, and optional MySQL export.

> Run the cells in order. The notebook uses the local CSV in `data/engineering_student_journey.csv` and writes the cleaned dataset to `data/cleaned_student_data.csv`.

In [ ]:
from pathlib import Path
import os
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Make paths reliable whether the notebook is opened from the project root or another folder.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "engineering_student_journey.csv").exists():
    PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
RAW_FILE = DATA_DIR / "engineering_student_journey.csv"
CLEANED_FILE = DATA_DIR / "cleaned_student_data.csv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 150

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data: {RAW_FILE}")
print(f"Output directory: {OUTPUT_DIR}")

## 1. Load and inspect the dataset

The source file is analyzed before any change is made. This helps identify malformed headers, missing values, duplicate records, and variable types.

In [ ]:
df = pd.read_csv(RAW_FILE)

print("Raw dataset dimensions:", df.shape)
print("Raw columns:")
for number, column in enumerate(df.columns, start=1):
    print(f"{number:02d}. {column}")

print("\nData types:")
print(df.dtypes.to_string())

print("\nMissing values:")
print(df.isna().sum()[lambda values: values > 0].to_string())

print("Duplicate rows:", df.duplicated().sum())
print("First five records:")
print(df.head(5).to_string(index=False))

## 2. Clean and prepare the data

Cleaning includes normalized headers, removal of exact duplicate rows, whitespace repair, missing-value treatment, and conversion of numeric-like fields. The cleaned file is the single source used for all downstream analysis.

In [ ]:
def clean_data(frame: pd.DataFrame) -> pd.DataFrame:
    """Normalize and clean the student dataset."""
    clean_df = frame.copy()

    # Normalize names so downstream filters are reliable.
    clean_df.columns = (
        clean_df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )

    # Remove exact duplicate observations, while retaining the first occurrence.
    clean_df = clean_df.drop_duplicates()

    # Clean text fields.
    for column in clean_df.select_dtypes(include=["object", "string"]).columns:
        clean_df[column] = clean_df[column].astype("string").str.strip()
        clean_df[column] = clean_df[column].str.replace(r"\s+", " ", regex=True)

    # Fill numeric missing values with the median and categorical missing values with the mode.
    for column in clean_df.select_dtypes(include=np.number).columns:
        clean_df[column] = clean_df[column].fillna(clean_df[column].median())

    for column in clean_df.select_dtypes(include=["string", "object"]).columns:
        mode_value = clean_df[column].mode(dropna=True)
        if not mode_value.empty:
            clean_df[column] = clean_df[column].fillna(mode_value.iloc[0])

    # Convert columns that are at least 90% numeric-like.
    for column in clean_df.select_dtypes(include=["string", "object"]).columns:
        candidate = clean_df[column].astype("string").str.replace("%", "", regex=False)
        converted = pd.to_numeric(candidate, errors="coerce")
        numeric_ratio = converted.notna().mean()
        if numeric_ratio >= 0.90:
            clean_df[column] = converted

    clean_df.to_csv(CLEANED_FILE, index=False)
    return clean_df

clean_df = clean_data(df)

print("Cleaned dataset dimensions:", clean_df.shape)
print("Cleaned columns:")
print(clean_df.columns.tolist())
print("\nRemaining missing values:")
print(clean_df.isna().sum()[lambda values: values > 0].to_string())
print("Remaining duplicate rows:", clean_df.duplicated().sum())

## 3. Exploratory data analysis

This section produces reusable findings and charts for business review. It identifies placement status, academic performance, attendance, branch mix, internship participation, and potential data-quality outliers.

In [ ]:
def find_columns(frame: pd.DataFrame, keywords: list[str]) -> list[str]:
    normalized = {column.lower(): column for column in frame.columns}
    matches = []
    for keyword in keywords:
        matches.extend(column for column in frame.columns if keyword in column.lower())
    return list(dict.fromkeys(matches))

placement_columns = find_columns(clean_df, ["placement_status", "placement"])
gpa_columns = find_columns(clean_df, ["gpa", "cgpa"])
attendance_columns = find_columns(clean_df, ["attendance"])

print("Placement columns:", placement_columns)
print("GPA columns:", gpa_columns)
print("Attendance columns:", attendance_columns)

# Placement distribution
if placement_columns:
    placement_col = placement_columns[0]
    placement_counts = clean_df[placement_col].value_counts(dropna=False)
    print("\nPlacement distribution")
    print(placement_counts.to_string())
    placement_counts.plot(kind="bar", color="#2563eb", figsize=(8, 5))
    plt.title("Student Placement Distribution")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "01_placement_distribution.png", dpi=300)
    plt.show()

# Correlation heatmap for numeric variables
numeric_columns = clean_df.select_dtypes(include=np.number).columns.tolist()
if len(numeric_columns) > 1:
    correlation = clean_df[numeric_columns].corr()
    print("\nCorrelation matrix")
    print(correlation.round(2).to_string())
    figure, axis = plt.subplots(figsize=(12, 8))
    sns.heatmap(correlation, annot=True, cmap="coolwarm", fmt=".2f", ax=axis)
    axis.set_title("Correlation Matrix")
    figure.tight_layout()
    figure.savefig(OUTPUT_DIR / "02_correlation_heatmap.png", dpi=300)
    plt.show()

# Academic and attendance distributions
for column in numeric_columns:
    if column in ["age", "backlogs"] or "gpa" in column.lower() or "attendance" in column.lower():
        figure, axis = plt.subplots(figsize=(8, 5))
        sns.histplot(data=clean_df, x=column, kde=True, ax=axis)
        axis.set_title(f"Distribution of {column.replace('_', ' ').title()}")
        axis.set_xlabel(column.replace("_", " ").title())
        axis.set_ylabel("Number of Students")
        figure.tight_layout()
        figure.savefig(OUTPUT_DIR / f"distribution_{column}.png", dpi=300)
        plt.show()

# Branch and placement cross-tabulation
branch_columns = find_columns(clean_df, ["branch"])
if branch_columns and placement_columns:
    branch_col = branch_columns[0]
    placement_col = placement_columns[0]
    branch_placement = pd.crosstab(
        clean_df[branch_col], clean_df[placement_col], normalize="index"
    ) * 100
    print("\nBranch placement rates (percentage)")
    print(branch_placement.round(2).to_string())

# Student-level relationship charts
if gpa_columns and attendance_columns and placement_columns:
    gpa_col = gpa_columns[0]
    attendance_col = attendance_columns[0]
    placement_col = placement_columns[0]

    print("\nAverage GPA by placement status")
    print(clean_df.groupby(placement_col)[gpa_col].agg(["mean", "median"]).round(2).to_string())

    figure, axis = plt.subplots(figsize=(8, 5))
    sns.boxplot(data=clean_df, x=placement_col, y=gpa_col, ax=axis)
    axis.set_title("GPA Distribution by Placement Status")
    axis.set_xlabel("Placement Status")
    axis.set_ylabel(gpa_col.replace("_", " ").title())
    figure.tight_layout()
    figure.savefig(OUTPUT_DIR / "gpa_by_placement.png", dpi=300)
    plt.show()

## 4. Business analytics KPIs

The following metrics support operational and strategic decisions, including enrollment quality, placement efficiency, academic performance, internship participation, and branch-level outcomes.

In [ ]:
# Find key fields dynamically.
placement_col = placement_columns[0] if placement_columns else None
gpa_col = gpa_columns[0] if gpa_columns else None
attendance_col = attendance_columns[0] if attendance_columns else None
branch_col = branch_columns[0] if branch_columns else None

if placement_col is None or gpa_col is None:
    raise ValueError("The dataset must contain placement_status and average_gpa columns.")

total_students = len(clean_df)
placed_students = int((clean_df[placement_col].astype(str).str.lower() == "placed").sum())
not_placed_students = total_students - placed_students
placement_rate = placed_students / total_students * 100 if total_students else 0
average_gpa = clean_df[gpa_col].mean()
median_gpa = clean_df[gpa_col].median()
average_attendance = clean_df[attendance_col].mean() if attendance_col else np.nan

# Internship participation and branch metrics
internship_columns = find_columns(clean_df, ["internship_done", "internship"])
internship_col = internship_columns[0] if internship_columns else None
internship_rate = 0
if internship_col:
    internship_yes = clean_df[internship_col].astype(str).str.lower().isin(["yes", "true", "1"])
    internship_rate = internship_yes.mean() * 100

kpis = {
    "total_students": total_students,
    "placed_students": placed_students,
    "not_placed_students": not_placed_students,
    "placement_rate": placement_rate,
    "average_gpa": average_gpa,
    "median_gpa": median_gpa,
    "average_attendance": average_attendance,
    "internship_participation_rate": internship_rate,
}

print("Business KPI summary")
for key, value in kpis.items():
    print(f"{key}: {value}")

# Branch-level placement performance
if branch_col:
    branch_summary = (
        clean_df.groupby(branch_col, dropna=False)
        .agg(
            total_students=("student_id", "size"),
            placed_students=(placement_col, lambda values: (values.astype(str).str.lower() == "placed").sum()),
            average_gpa=(gpa_col, "mean"),
            average_attendance=(attendance_col, "mean") if attendance_col else (attendance_col, "mean"),
        )
        .assign(placement_rate=lambda frame: frame["placed_students"] / frame["total_students"] * 100)
        .sort_values("placement_rate", ascending=False)
    )
    print("\nBranch performance")
    print(branch_summary.round(2).to_string())
    branch_summary.to_csv(OUTPUT_DIR / "branch_performance.csv")

# Save business KPI values in a machine-readable JSON file.
kpis_json = {
    key: float(value) if isinstance(value, np.floating) else value
    for key, value in kpis.items()
}
with open(OUTPUT_DIR / "business_kpis.json", "w", encoding="utf-8") as file:
    json.dump(kpis_json, file, indent=2)

## 5. Business interpretation

Use the results to answer questions such as:

- Which branches have the strongest placement outcomes?
- Does higher average GPA correlate with placement success?
- Is attendance associated with academic or placement performance?
- Which student skills or clubs are linked to internship and placement outcomes?
- Which academic or behavioral indicators should be used in student support programs?

In [ ]:
# Create a concise, decision-oriented report for business and academic stakeholders.
report = {
    "dataset_name": "Engineering Student Journey",
    "total_students": total_students,
    "placement_rate_percent": round(placement_rate, 2),
    "average_gpa": round(float(average_gpa), 2),
    "average_attendance_percent": round(float(average_attendance), 2) if pd.notna(average_attendance) else None,
    "internship_participation_rate_percent": round(internship_rate, 2),
    "business takeaway": (
        "Placement performance is the primary outcome; compare branch-level placement rates, "
        "academic scores, attendance, internships, and skill profiles before designing support."
    ),
}

with open(OUTPUT_DIR / "executive_summary.json", "w", encoding="utf-8") as file:
    json.dump(report, file, indent=2)

print("Executive summary saved to outputs/executive_summary.json")

## 6. Optional MySQL export

The cleaned dataset can be loaded into MySQL. Set the environment variables in the same terminal before running this cell. The table name is `students`, and the operation uses `CREATE TABLE IF NOT EXISTS` plus transactional inserts.

In [ ]:
def load_cleaned_data_to_mysql() -> None:
    """Load cleaned data to MySQL when credentials are configured."""
    from dotenv import load_dotenv
    import mysql.connector

    load_dotenv()
    required_variables = ["MYSQL_HOST", "MYSQL_PORT", "MYSQL_USER", "MYSQL_PASSWORD", "MYSQL_DATABASE"]
    missing = [name for name in required_variables if not os.getenv(name)]

    if missing:
        print("MySQL export skipped. Missing variables:", ", ".join(missing))
        print("Set these variables in a .env file or your environment before running this cell.")
        return

    connection = mysql.connector.connect(
        host=os.getenv("MYSQL_HOST"),
        port=int(os.getenv("MYSQL_PORT", "3306")),
        user=os.getenv("MYSQL_USER"),
        password=os.getenv("MYSQL_PASSWORD"),
        database=os.getenv("MYSQL_DATABASE"),
    )
    cursor = connection.cursor()

    column_definitions = []
    for column in clean_df.columns:
        dtype = clean_df[column].dtype
        if pd.api.types.is_integer_dtype(dtype):
            sql_type = "INT"
        elif pd.api.types.is_float_dtype(dtype):
            sql_type = "DOUBLE"
        else:
            sql_type = "VARCHAR(255)"
        column_definitions.append(f"`{column}` {sql_type}")

    create_table_query = (
        "CREATE TABLE IF NOT EXISTS students ("
        + ", ".join(column_definitions)
        + " )"
    )
    cursor.execute(create_table_query)

    column_names = ", ".join(f"`{column}`" for column in clean_df.columns)
    placeholders = ", ".join(["%s"] * len(clean_df.columns))
    insert_query = f"INSERT INTO students ({column_names}) VALUES ({placeholders})"

    rows = []
    for row in clean_df.itertuples(index=False, name=None):
        rows.append(tuple(None if pd.isna(value) else value for value in row))

    cursor.executemany(insert_query, rows)
    connection.commit()
    cursor.close()
    connection.close()
    print(f"Loaded {len(rows)} rows into MySQL table students.")

# Uncomment the next line only after configuring MySQL credentials.
# load_cleaned_data_to_mysql()


## 7. Complete workflow checklist

1. Run the imports and dataset inspection cell.
2. Run the cleaning cell to regenerate `data/cleaned_student_data.csv`.
3. Run the EDA and KPI cells to generate reports and charts.
4. Review `outputs/business_kpis.json`, `outputs/branch_performance.csv`, and `outputs/executive_summary.json`.
5. Configure MySQL credentials only if database export is required.